# Tap&Go Fraud Detection: Exploratory Data Analysis (EDA)

This notebook explores the synthetic transaction dataset for the Tap&Go platform. Our goal is to understand the underlying patterns of legitimate vs. fraudulent transactions before feeding them into our Machine Learning pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load the Data
We will load the training split (`tapgo_train.csv`) to explore the data. It is best practice to only perform EDA on the training set to avoid leaking information from the validation or test sets.

In [ ]:
df = pd.read_csv('../data/tapgo_train.csv')
df['transaction_timestamp'] = pd.to_datetime(df['transaction_timestamp'])
df.head()

## 2. Class Imbalance (Fraud vs. Legitimate)
Fraud detection datasets are inherently heavily imbalanced. Let's look at the exact distribution of the `is_fraud` target variable.

In [ ]:
counts = df['is_fraud'].value_counts()
labels = ['Legitimate (0)', 'Fraudulent (1)']

plt.figure(figsize=(6,6))
plt.pie(counts, labels=labels, autopct='%1.2f%%', colors=['#4C72B0', '#C44E52'], explode=[0, 0.1])
plt.title('Class Distribution: Legitimate vs. Fraud')
plt.show()

## 3. Transaction Amount Analysis
Do fraudsters spend more money than legitimate users? Let's compare the density distributions of transaction amounts for both classes.

In [ ]:
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df[df['is_fraud'] == 0], x='transaction_amount', label='Legitimate', fill=True, color='#4C72B0')
sns.kdeplot(data=df[df['is_fraud'] == 1], x='transaction_amount', label='Fraud', fill=True, color='#C44E52')
plt.title('Distribution of Transaction Amounts by Class')
plt.xlabel('Transaction Amount (₹)')
plt.ylabel('Density')
plt.xlim(0, df['transaction_amount'].quantile(0.99)) # Cut off extreme outliers for visibility
plt.legend()
plt.show()

## 4. Time-Based Patterns
Fraudsters often operate at different times of the day compared to regular commuters. Let's examine transaction frequencies by the hour of the day. We normalize the density so the two classes are comparable on the same scale.

In [ ]:
plt.figure(figsize=(12, 6))
sns.histplot(data=df, x='transaction_hour', hue='is_fraud', bins=24, multiple='dodge', stat='density', common_norm=False, palette=['#4C72B0', '#C44E52'])
plt.title('Normalized Transaction Frequency by Hour of Day')
plt.xlabel('Hour of Day (0-23)')
plt.ylabel('Density (Normalized per class)')
plt.xticks(range(0, 24))
plt.show()

## 5. Velocity and Feature Correlations
Finally, let's look at how heavily correlated our key engineered features (like `transactions_last_1hour`, `failed_otp_attempts`, and `location_change_rate`) are with the target `is_fraud` variable.

In [ ]:
features_to_correlate = [
    'is_fraud', 'transaction_amount', 'transactions_last_1hour', 
    'failed_otp_attempts', 'location_change_rate', 'new_device'
]

corr = df[features_to_correlate].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap of Key Fraud Signals')
plt.show()